In [2]:
%set_env AZURE_OPENAI_ENDPOINT=https://swedenopenairesource.openai.azure.com
%set_env AZURE_OPENAI_ENDPOINT_KEY=7e28b9e1a8cf492eabc27b5742da5aab
%set_env AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
%set_env AZURE_OPENAI_API_VERSION=2023-03-15-preview


env: AZURE_OPENAI_ENDPOINT=https://swedenopenairesource.openai.azure.com
env: AZURE_OPENAI_ENDPOINT_KEY=7e28b9e1a8cf492eabc27b5742da5aab
env: AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
env: AZURE_OPENAI_API_VERSION=2023-03-15-preview


In [14]:
import os
from openai import AzureOpenAI
    
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_ENDPOINT_KEY"),  
    api_version= os.getenv("AZURE_OPENAI_API_VERSION"), #"2024-02-01",
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    )
    
deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") #This will correspond to the custom name you chose for your deployment when you deployed a model. Use a gpt-35-turbo-instruct deployment. 

def process_json(json_str):    
    response = client.chat.completions.create(
        model=deployment_name, # model = "deployment_name".
        messages=[
            {"role": "system", "content": "You are a digital assistant tasked to help extract recipe names from json documents.  Find all words that might consitute a recipe name and respond as json array, for example ['italian', 'wedding', 'soup']"},
            {"role": "user", "content": json_str}
        ]
    )

    return response.choices[0].message.content

print(process_json("['<sleep>17.601534', '<image>screenshot_2024-08-31_11-34-45.374153', 'spaghetti', '<key>Key.space:True', 'carbonara', '<sleep>2.795768', '<image>screenshot_2024-08-31_11-34-50.991518', '<mouse>on_click(1399,191,Button.left,True)', '<sleep>2.84245', '<image>screenshot_2024-08-31_11-34-53.678416']	['<mouse>on_click(598,963,Button.left,True)', '<sleep>6.415703']"))



```json
["spaghetti", "carbonara"]
```


In [38]:
import json
import sys

keylog_dir = "C:\\Users\\antonslutsky\\Dev\\azureml-quickstart\\slm-finetuning\\phi3-vision-finetune\\applications\\output\\session_2024-08-31_11-24-25\\"
keylog_txt = "keylog.txt"

keylog_txt_aug = "keylog_aug.txt"

keylog_path = keylog_dir + keylog_txt

keylog_aug_path = keylog_dir + keylog_txt_aug



with open(keylog_path, "r") as keylog_file, open(keylog_aug_path, "w") as keylog_aug_file:
    lines = keylog_file.read().split("\n")
    print("Read lines:", len(lines))
    for line in lines:
        try:
            gpt_response = process_json(line)
            print(f"gpt_response: {gpt_response}")
            gpt_json = gpt_response.replace("json", "").replace("```", "").replace("\'", "\"").strip()
            print(f"------ gpt_json {gpt_json} -----------")
            gpt_json = json.loads(gpt_json)
            #print(f"gpt_json: {gpt_json}")
            if len(gpt_json) > 0:
                target_name = " ".join(gpt_json)
                print(target_name)
                prompt = f"""You are a useful AI that searches AllRecipes.com website for various recipies.  The following json document contains a set of keyboard and mouth actions together with the screenshots that preempted them to search for '{target_name}' recipe on the website.  Suggest the nest set of keyboard and mouth actions to continue searching for the recipe. {line}"""
                print(prompt)
                keylog_aug_file.write(prompt+"\n")

        except json.JSONDecodeError as e:
            print(f"Failed to process line: {line}", e)


Read lines: 93
gpt_response: ```json
[]
```
------ gpt_json [] -----------
gpt_response: ```json
["italian", "wedding", "soup"]
```
------ gpt_json ["italian", "wedding", "soup"] -----------
italian wedding soup
You are a useful AI that searches AllRecipes.com website for various recipies.  The following json document contains a set of keyboard and mouth actions together with the screenshots that preempted them to search for 'italian wedding soup' recipe on the website.  Suggest the nest set of keyboard and mouth actions to continue searching for the recipe. ['<sleep>11.645689', '<image>screenshot_2024-08-31_11-24-34.961750', '<mouse>on_click(1070,182,Button.left,True)', '<sleep>3.881456', '<image>screenshot_2024-08-31_11-24-40.230237', 'italian', '<key>Key.space:True', 'wedding', '<key>Key.space:True']	['soup', '<sleep>1.752508']
gpt_response: ```json
["italian", "wedding", "soup"]
```
------ gpt_json ["italian", "wedding", "soup"] -----------
italian wedding soup
You are a useful AI 